# Exploring the Standard Model

Written by:

* Philip Ilten (University of Cincinnati)

This notebook provides a short introduction to some of the underlying physics of the Standard Model (SM) of particle physics: conservation of momentum, particle kinematics, electromagnetic coupling, and quantum chromodynamics radiation.

## Requirements

Before running this notebook, we need to set up our environment. First, we install and import the `wurlitzer` module. This allows programs that have C-like backends to write their output to the Python console. In short, this allows the output of Pythia to be displayed in this notebook.

In [ ]:
# Redirect the C output of Pythia to the notebook.
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

Next, we need to install the Pythia module.

In [ ]:
# Install and import the Pythia module.
!pip install pythia8mc
import pythia8mc as pythia8

Finally, we need to download and import the visualization module `vistas`.

In [ ]:
# Download the visualization module.
!wget -q -N https://gitlab.com/mcgen-ct/tutorials/-/raw/main/.full/pythia/vistas.py

# Import `vistas` module.
import vistas

_oldStrength = vistas.Vistas.strength
_oldChroma = vistas.Vistas.chroma
_oldColor = vistas.Vistas.color

We also need to set up the visualization. The following two cells modify the visualization for this notebook so things are a little more clear.

In [ ]:
# Allow color by PDG ID.
def _newStrength(self, prt, att, cat):
    import math

    if att["observable"] == "p.idAbs()":
        pid = prt.idAbs()
        pids = [1, 2, 3, 4, 5, 11, 13, 15, 23]
        if pid in pids:
            return pids.index(pid) / len(pids)
        else:
            return 1
    return _oldStrength(self, prt, att, cat)


def _newChroma(t, color):
    return _oldColor(t)

In [ ]:
# If a web browser window is not opening, you can set this to a higher
# value (in seconds) and then click the printed link.
sleep = 1


# The following creates a VISTAS object and configures it.
def Viewer(pythia, show=None):
    if show is None:
        vistas.Vistas.strength = _newStrength
        vistas.Vistas.chroma = _newChroma
    else:
        vistas.Vistas.strength = _oldStrength
        vistas.Vistas.chroma = _oldChroma
    viewer = vistas.Vistas(pythia)
    viewer.opts = viewer.options()
    viewer.opts["length"]["scale"] = "log"
    viewer.opts["length"]["observable"] = "p.pAbs()"
    if show is None:
        show = ["hard process"]
        viewer.opts["color"]["observable"] = "p.idAbs()"
        viewer.opts["color"]["min"] = -1
        viewer.opts["color"]["max"] = 1
    if len(show) > 0:
        viewer.opts["show"] = show
    viewer.opts["frame"] = "lab"
    viewer.opts["length"]["skew"] = 1
    viewer.opts["length"]["offset"] = 30
    return viewer

## Introduction

The Standard Model (SM) of particle physics describes what we believe to be the interactions between fundamental particles, objects that have no known substructure. The Pythia event generator implements the math behind the SM and simulates the interactions of these particles. Conceptually, using Pythia is straightforward; we tell Pythia what our initial particles are, and it then generates the resulting particles from these colliding particles. These final particles are what would be observed by, for example, the large detectors at the European Organization for Nuclear Research (CERN) such as ATLAS, CMS, or LHCb on the Large Hadron Collider (LHC).

Before we get started, let's perform a simulation for the production of a Higgs boson at the LHC. This will give us an idea of the complexity of these simulations. However, in this notebook, we will study processes that are less complex so we can better understand the physics.

In [ ]:
# Create the Pythia simulation object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")
pythia.readString("Random:setSeed = on")
pythia.readString("Random:seed = 1")

# Define the initial particles.
# Beam A is a proton.
pythia.readString("Beams:idA = 2212")
# Beam B is a proton.
pythia.readString("Beams:idB = 2212")

# Set the initial particles center-of-mass energy (13 TeV).
pythia.readString("Beams:eCM = 13000")

# Tell Pythia what process we want to simulate.
# This turns on all Higgs production.
pythia.readString("HiggsSM:all = on")

# Initialize Pythia.
pythia.init()

# Simulate the collision.
pythia.next()

# View the result.
viewer = vistas.Vistas(pythia)
viewer.display(sleep=sleep)

A new window should open which displays the simulation. If this window doesn't open, make sure pop-ups are allowed for this notebook or click the printed link before the cell finishes executing. If a detector displays, rather than the simulation, try running the cell again. Each line in the display corresponds to a particle produced in the simulation.

Let's now switch to a less complex process, and try simulating an electron and positron colliding at a relatively low energy with Pythia. First, we need to tell Pythia what we want to simulate.

In [ ]:
# Create the Pythia simulation object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")
pythia.readString("Random:setSeed = on")
pythia.readString("Random:seed = 1")

# Define the initial particles.
# Beam A is an electron.
pythia.readString("Beams:idA = 11")
# Beam B is a positron.
pythia.readString("Beams:idB = -11")

# Allow Pythia to simulate low-energy collisions.
pythia.readString("PhaseSpace:mHatMin = 0.0")
pythia.readString("23:mMin = 0.0")

# Set the momentum of the initial particles.
pythia.readString("Beams:frameType = 3")
# Give the electron a momentum of 0.4 GeV along the positive z-direction.
pythia.readString("Beams:pxA = 0")
pythia.readString("Beams:pyA = 0")
pythia.readString("Beams:pzA = 0.4")
# Give the positron a momentum of 0.4 GeV along the negative z-direction.
pythia.readString("Beams:pxB = 0")
pythia.readString("Beams:pyB = 0")
pythia.readString("Beams:pzB = -0.4")

# Tell Pythia what process we want to simulate.
# This process is e+ e- -> gamma*/Z: gamma* is a virtual photon,
# and Z a Z0 boson. Below is the Feynman diagram for this process.
#  e-                    ?
#     \                /
#      \              /
#       \            /
#        *~~~~~~~~~~*
#       /  gamma*/Z  \
#      /              \
#     /                \
#  e+                    ?
pythia.readString("WeakSingleBoson:ffbar2gmZ = on")

# Turn off some parts of the simulation that aren't needed yet.
# Specifically, this turns off everything except the initial collision.
pythia.readString("PDF:lepton = off")
pythia.readString("PartonLevel:ISR = off")
pythia.readString("PartonLevel:FSR = off")
pythia.readString("HadronLevel:Hadronize = off")
pythia.readString("HadronLevel:Decay = off")

# Initialize Pythia.
pythia.init()

Next we can visualize the simulation produced by Pythia. The following cell should open up a new browser window. If it does not, click the printed link.

In [ ]:
# Simulate the collision.
pythia.next()

# View the result.
viewer = Viewer(pythia)
viewer.display(sleep=sleep)

Each line represents a particle, where the direction of the line is the direction of the particle's momentum, and the length of the line is the magnitude of the particle's momentum. Use the object selection tool to inspect each particle (looks like a mouse cursor hovering over a circle). Identify the initial electron and positron in the simulation. What are the final particles that are produced?

###START_SOLUTION As discussed in further detail below, the final state depends on the random state of Pythia. However, given the center-of-mass energy of the system, also explored more below, we should see final states of $e^+e^-$, $\mu^+ \mu^-$, $d\bar{d}$, and $u\bar{u}$.
###STOP_SOLUTION

What's going on with the particle line that connects the initial and final particles? This is the intermediate particle that Pythia simulates which describes the force through which this interaction is occuring. Use the selection tool to determine what type of particle this intermediate particle is. Is this particle the carrier for one of the four forces of the SM? If so, which one?

###START_SOLUTION The particle is labeled as `Z0` which is a $Z^0$ boson, a carrier of the weak force. Actually, this intermediate particle is a quantum mechanical mixture of both a virtual photon, $\gamma^*$, and the $Z^0$ boson. Pythia uses the label `Z0` as shorthand for this. At this center-of-mass energy, this intermediate particle behaves like a virtual photon as it is very far from the nominal mass of the $Z^0$.
###STOP_SOLUTION

If we were to run the simulation cell above again, would we get the same result? Try this. Compare to how this simulation might be similar or different to colliding two pool balls together.

###START_SOLUTION Even if the initial state is exactly the same, we will get different final states, including the momentum vectors. In nature, this is because quantum mechanics is probabilistic. In the simulation, this is because Pythia uses pseudo-random number techniques to simulate different final states each time the simulation is called. This type of numerical technique, using pseudo-random number generation, is called Monte Carlo.
###STOP_SOLUTION

## Kinematics

In physics we use the word kinematics to describe the movement of objects. In particle physics, kinematics specifically means the momentum-related observables for a particle. Run the simulation cell a few more times. Can you observe any connection between the kinematics of the two outgoing particles for each simulation? Use the selection tool to find the numerical value for each component of the final particles' momentum ($p_x$, $p_y$, and $p_z$). Can you show what the connection is with these numbers?

###START_SOLUTION In every simulation we should see that for each component of momentum ($p_x$, $p_y$, and $p_z$), the sum of that component should be zero. Visually, we see this manifest as the two final particles are always pointing back-to-back. Physically, this means that momentum is conserved in this simulation.
$$
\begin{aligned}
p_{x,1} + p_{x,2} &= 0 \\
p_{y,1} + p_{y,2} &= 0 \\
p_{z,1} + p_{z,2} &= 0 \\
\end{aligned}
$$
###STOP_SOLUTION

We have set up our initial electron and positron to have momentum in equal but opposite directions along the $z$-axis. What happens if we change the momentum of the electron to be larger?

In [ ]:
# Change the electron's momentum along the positive z-direction.
pythia.readString("Beams:pzA = 1")
pythia.init()
viewer = Viewer(pythia)

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

The intermediate particle now covers up the initial positron so we can no longer see it. Does the connection between the momentum components of the final particles from before still hold? What about a connection between the summed momenta of the initial particles and the final particles? What about a connection between the summed energy of the initial particles and the final particles?

###START_SOLUTION We still have conservation of momentum, but our initial momentum is no longer zero, with a net momentum along the $z$-axis. Visually we can see this, where our momentum along the $x$- and $y$-axis still balances out to zero, but not along the $z$-axis.
$$
\begin{aligned}
p_{x,1} + p_{x,2} &= 0 \\
p_{y,1} + p_{y,2} &= 0 \\
p_{z,1} + p_{z,2} &= p_{z,e^-} + p_{z,e^+} \\
\end{aligned}
$$
If we inspect the energy of the initial particles and final particles, we also see we have conservation of energy.
$$
E_1 + E_2 = E_{e^-} + E_{e^+}
$$
###STOP_SOLUTION

In special relativity there is a relation between energy and momentum, the energy-momentum relation.
$$
E^2 = (pc)^2 + (mc^2)^2
$$
This is the more complete version of the famous $E = mc^2$ equation. Practically, this means that if a particle has a given mass we can trade momentum for energy, or vice versa. We can already see this in the simulations we performed. Use the selection tool and the intermediate particle to confirm that Pythia is respecting the energy-momentum relation.

Increase the momentum of the electron in the simulation cell above even more by changing `Beams:pzA`. Run a few simulations. On average does the angle between the final particles change from the lower-momentum simulation? How might this be related to the energy-momentum relation?

###START_SOLUTION In particle physics we typically call this an opening angle. As we increase the momentum of the electron but keep the momentum of the positron the same, we see that the opening angle becomes smaller. This is due to both the energy-momentum relation and conservation of energy and momentum. As the summed initial particle $p_z$ becomes larger, the $p_z$ of the final particles must also shift in this direction due to conservation of momentum, which closes this opening angle.
###STOP_SOLUTION

## Couplings

In the previous section we saw that not every simulation we ran had the same final particles. Let's see if we can find a pattern in the types of these final particles that are produced.

In [ ]:
# We need to go back to our initial set up.
pythia.readString("Beams:pzA = 0.4")
# Change this seed to a unique number if running with other groups.
pythia.readString("Random:seed = 1")
pythia.init()
viewer = Viewer(pythia)

Now, run the simulation below at a minimum twenty times (ideally more like one hundred times) and track the number of different final particle states that are produced. If working with other groups, make sure to set `Random:seed` to a unique number for each group.

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

What are the counts of the different final particle states? Which final particle state is the most common? Which is the least common? Are there any final particles states which have similar counts?

The process we are simulating occurs through a virtual photon. The photon couples to a specific final particle state with a strength given by the product of the absolute value of the electromagnetic charges of the particles.

| fermion | charge |
| --- | --- |
| $e^-$ | $-1$ |
| $\mu^-$ | $-1$ |
| $u$ | $+2/3$ |
| $d$ | $-1/3$ |

Using the table above, do the simulated number of particle counts make sense? If not, what might be missing?

###START_SOLUTION The cell below automates the counting by accessing the event record of Pythia directly to determine the type of final particles, but should produce the same result as inspecting visually. We see that the photon does indeed couple to the product of charge, but notice that the quark final states are multiplied by a factor of three, corresponding to the three different color charges they can carry.
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
counts = {}
total = 1000
# Run `total` simulations.
for sim in range(0, total):
    # Simulate.
    pythia.next()
    # Get the final state.
    pids = [pythia.event[dtr].id() for dtr in pythia.event[5].daughterList()]
    # Sort and create a key.
    pids = tuple(sorted(pids))
    # Increment the final particle state.
    counts[pids] = counts.get(pids, 0) + 1
# Determine the predicted total.
pred, predsum = {}, 0
for pids in counts.keys():
    # Get the particle information.
    pde = pythia.particleData.findParticle(pids[0])
    # Determine the factor.
    factor = pde.charge() ** 2 * (3 if pde.colType() == 1 else 1)
    # Set the factor and add to the total prediction.
    pred[pids] = factor
    predsum += factor
# Print the result.
print(" counts   pred.   particles")
for pids, count in sorted(counts.items(), key=lambda item: item[1], reverse=True):
    # Get the final state name.
    names = [pythia.particleData.findParticle(pid).name(pid) for pid in pids]
    print(f"{count:>7d} {total*pred[pids]/predsum:>7.2f}   {' '.join(names)}")
###STOP_SOLUTION

## Thresholds

So far we have only seen four final particle states: $e^+e^-$, $\mu^+\mu^-$, $u\bar{u}$, and $d\bar{d}$; why aren't the other quarks and leptons that we know exist being produced? Increase the momentum of both the initial electron and positron to 1.7 GeV and run the simulation a few times. Are there any new final particle states? What happens if the electron and positron momentum is increased to 2 GeV and then 5 GeV?

In [ ]:
# Set up our new beams.
pythia.readString("Beams:pzA = 1.7")
pythia.readString("Beams:pzB = -1.7")
pythia.init()
viewer = Viewer(pythia)

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

###START_SOLUTION Increasing the momentum to 1.7 GeV, we begin to see both the $s\bar{s}$ and $c\bar{c}$ final state. Further increasing the momentum to $2$ GeV we see the $\tau^+\tau^-$ final state, and increasing the momentum to 5 GeV we see the $b\bar{b}$ final state.
###STOP_SOLUTION

Look up the masses (in GeV) of the $\tau$ lepton, the charm quark ($c$), and the beauty (or bottom) quark ($b$). Is there a connection between these masses, the final particle states we observe, and the momentum of the initial electron and positron?

###START_SOLUTION We see that the summed energy of the electron and positron must be larger than the summed masses of the final particles. This is because of the energy-momentum relation. We must have sufficient energy in the initial particles to trade for mass in the final particles. What about the top quark ($t$)? Due to a number of details behind the simulation of top quarks, Pythia simulates these with an independent process turned on with the setting `Top:ffbar2ttbar(s:gmZ) = on`. So with the Pythia configuration we have here, even setting the momentum above 200 GeV will not produce top quarks in the simulation.
###STOP_SOLUTION

## Quantum Chromodynamics

In our simulation above, we have final particle states that have quarks in them, like $u \bar{u}$. But, we have never observed a quark by itself in nature; they are always part of a bound state like a proton. So what is happening in our simulation? We have turned off two important parts of the simulation. Let's turn them back on, one-by-one.

The first missing part of the simulation is particle radiation. Just like charged particles like an electron can radiate photons, quarks can radiate gluons. We can turn this back on with the following cell.

In [ ]:
# Increase the collision energy.
pythia.readString("Beams:pzA = 20")
pythia.readString("Beams:pzB = -20")
# Turn on final state radiation.
pythia.readString("PartonLevel:FSR = on")
pythia.init()
viewer = Viewer(pythia, ["hard process", "FSR"])

Run a few simulations. What has changed in the simulation? Are there any general patterns in the radiation off the quarks that is now included?

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

###START_SOLUTION For final particle states where there are quarks we should now see radiation from these quarks. We can observe a few general patterns in these simulations.

1. The radiation is typically not just a single gluon but rather a "shower" of quarks and gluons. This is because the coupling of the strong force can be large, which will cause a significant amount of radiation.
2. The types of radiation we see is $q \to q g$, $\bar{q} \to \bar{q} g$, $g \to g g$, or $g \to q \bar{q}$. This is due to the underlying theory of quantum chromodynamics (QCD).
3. The radiation is usually, but not always, roughly along the same direction of the initial quark that is radiating. Again, this is due to the underlying theory of QCD which prefers radiation along the same direction ("collinear") or very low-energy radiation ("soft"). This also is because of the boost of this radiation.

In particle physics, we call these collimated sprays of quarks and gluons "jets". Measuring these jets with experiments help us understand the underlying QCD that produces them.

###STOP_SOLUTION

The second part of the simulation that is missing is the combination of quarks and gluons into bound states into hadrons. This process is called "hadronization". We can turn this part of the simulation on with the cell below.

In [ ]:
# Turn on hadronization.
pythia.readString("HadronLevel:Hadronize = on")
pythia.init()
viewer = Viewer(pythia, [])

Run a few simulations. What are some of the hadrons that are being produced? Are there any patterns in the production of these hadrons?

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

###START_SOLUTION The cell below automates the counting by accessing the event record of Pythia directly to count the types of final particles. We see that the $\pi$, $\omega$, $\rho$, and $\eta$ mesons (both charged and neutral) are the most common hadrons produced. Viewing a few of these events, we also can see that the hadrons tend to be formed along the same direction as the quarks and gluons producing them. This is called local parton-hadron duality and preserves the shape of the jets.
###STOP_SOLUTION

In [ ]:
###START_SOLUTION
counts = {}
total = 1000
# Run `total` simulations.
for sim in range(0, total):
    # Simulate.
    pythia.next()
    # Loop over the final particles.
    for prt in pythia.event:
        if not prt.isFinal():
            continue
        # Increment the final particle type.
        counts[prt.id()] = counts.get(prt.id(), 0) + 1
# Print the result.
print(" counts   particle")
for pid, count in sorted(counts.items(), key=lambda item: item[1], reverse=True):
    # Get the final state name.
    name = pythia.particleData.findParticle(pid).name(pid)
    print(f"{count:>7d}   {name}")
###STOP_SOLUTION